# 06 — Batch Inference

Scores a small sample of "new applications" (raw rows with the target column dropped) using the latest trained model — exactly what `poetry run bnpl-risk predict-batch` does nightly. See `docs/inference.md`.

In [ ]:
from pathlib import Path

from bnpl_credit_risk.data.loaders import BNPLDataLoader
from bnpl_credit_risk.pipelines.batch_inference_pipeline import run_batch_inference_pipeline
from bnpl_credit_risk.settings import get_settings, load_config

settings = get_settings()
config = load_config()
df = BNPLDataLoader(settings, config.data).load_raw()
applications = df.drop(columns=[config.data.target_column]).sample(30, random_state=1)
input_path = Path('../data/processed/notebook_applications_to_score.csv').resolve()
input_path.parent.mkdir(parents=True, exist_ok=True)
applications.to_csv(input_path, index=False)

In [ ]:
output_path = Path('../data/predictions/notebook_predictions.csv').resolve()
result = run_batch_inference_pipeline(config, settings, input_path=input_path, output_path=output_path)
result.n_scored, result.n_rejected, result.risk_band_distribution

In [ ]:
import pandas as pd

predictions = pd.read_csv(result.output_path)
predictions.head()

In [ ]:
predictions['risk_band'].value_counts().plot(kind='bar', title='Risk band distribution')